In [3]:
import os
import glob
import pandas as pd
import numpy as np
import anndata as ad
from scipy import sparse


In [68]:
def build_anndata_from_path(path):
    # ---------- Load abundance ----------
    abund = pd.read_csv(path)
    
    obs = abund.select_dtypes(exclude="number").astype(str)
    if "Age" in abund.columns:
        obs = obs.join(abund[["Age"]])
    
    # set obs index
    obs.index = abund["Study_ID"].astype(str)
    obs.index.name = "study_id"
    
    X_df = abund.drop(columns=["Age"], errors="ignore").select_dtypes(include="number").astype(float)
    
    # ---------- Build AnnData ----------
    adata = ad.AnnData(
        X=sparse.csr_matrix(X_df.values),
        obs=obs,
        var=pd.DataFrame(index=X_df.columns)
    )


    print(f"final shape: {adata.shape}")

    return adata


In [69]:
longitudinal = build_anndata_from_path("/home/kchen/microbiome/gut_microbiome_GPT/datasets/gmwi2_csv/longitudinal_cases.csv")
cross_sectional = build_anndata_from_path("/home/kchen/microbiome/gut_microbiome_GPT/datasets/gmwi2_csv/Test_dataset_metaphlan3.csv")
training = build_anndata_from_path("/home/kchen/microbiome/gut_microbiome_GPT/datasets/gmwi2_csv/training_set.csv")

/tmp/ipykernel_140594/130557936.py:3: DtypeWarning: Columns (3204) have mixed types. Specify dtype option on import or set low_memory=False.
  abund = pd.read_csv(path)
/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


final shape: (589, 3200)


/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


final shape: (1140, 2084)


/tmp/ipykernel_140594/130557936.py:3: DtypeWarning: Columns (3204,3205) have mixed types. Specify dtype option on import or set low_memory=False.
  abund = pd.read_csv(path)


final shape: (8069, 3200)


/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [70]:
print(longitudinal, cross_sectional, training)

AnnData object with n_obs × n_vars = 589 × 3200
    obs: 'Study_ID', 'Sample Accession', 'Subject_ID', 'health_status/host_diet', 'timepoint' AnnData object with n_obs × n_vars = 1140 × 2084
    obs: 'Study_ID', 'Sample Accession', 'is_healthy', 'Continent', 'Phenotype' AnnData object with n_obs × n_vars = 8069 × 3200
    obs: 'Study_ID', 'Sample Accession', 'is_healthy', 'Sex', 'Continent', 'Phenotype', 'Age'


In [71]:
import anndata as ad

adatas = [longitudinal, cross_sectional, training]

# 1) inner-merge obs columns (intersection)
common_obs_cols = sorted(set.intersection(*(set(a.obs.columns) for a in adatas)))

# 2) subset obs to common columns
adatas = [
    a.copy() if common_obs_cols == list(a.obs.columns) else a.copy()
    for a in adatas
]
for a in adatas:
    a.obs = a.obs[common_obs_cols].copy()

# 3) outer-merge vars (union)
adata_merged = ad.concat(
    adatas,
    axis=0,
    join="outer",      # union of var/features
    index_unique=None, # keep obs_names
    fill_value=0,      # missing taxa -> 0 (good for sparse abundance tables)
)

print("common obs cols:", common_obs_cols)
print(adata_merged)


common obs cols: ['Sample Accession', 'Study_ID']
AnnData object with n_obs × n_vars = 9798 × 3434
    obs: 'Sample Accession', 'Study_ID'


/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/home/kchen/miniconda3/envs/hmbGPT/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [72]:
adata_merged

AnnData object with n_obs × n_vars = 9798 × 3434
    obs: 'Sample Accession', 'Study_ID'

In [73]:
datas = [longitudinal, cross_sectional, training, adata_merged]
names = ["gmwi_longitudinal", "gmwi_cross_sectional", "gmwi_training", "gmwi_merged"]
for data, name in zip(datas, names):
    print(data)
    data.write_h5ad(f"/home/kchen/microbiome/gut_microbiome_GPT/datasets/metagenomics/{name}.h5ad")

longitudinal.var_names
    

AnnData object with n_obs × n_vars = 589 × 3200
    obs: 'Study_ID', 'Sample Accession', 'Subject_ID', 'health_status/host_diet', 'timepoint'
AnnData object with n_obs × n_vars = 1140 × 2084
    obs: 'Study_ID', 'Sample Accession', 'is_healthy', 'Continent', 'Phenotype'
AnnData object with n_obs × n_vars = 8069 × 3200
    obs: 'Study_ID', 'Sample Accession', 'is_healthy', 'Sex', 'Continent', 'Phenotype', 'Age'
AnnData object with n_obs × n_vars = 9798 × 3434
    obs: 'Sample Accession', 'Study_ID'


Index(['k__Archaea', 'k__Archaea|p__Euryarchaeota',
       'k__Archaea|p__Euryarchaeota|c__Methanobacteria',
       'k__Archaea|p__Euryarchaeota|c__Methanobacteria|o__Methanobacteriales',
       'k__Archaea|p__Euryarchaeota|c__Methanobacteria|o__Methanobacteriales|f__Methanobacteriaceae',
       'k__Archaea|p__Euryarchaeota|c__Methanobacteria|o__Methanobacteriales|f__Methanobacteriaceae|g__Methanobrevibacter',
       'k__Archaea|p__Euryarchaeota|c__Methanobacteria|o__Methanobacteriales|f__Methanobacteriaceae|g__Methanobrevibacter|s__Methanobrevibacter_smithii',
       'k__Archaea|p__Euryarchaeota|c__Methanobacteria|o__Methanobacteriales|f__Methanobacteriaceae|g__Methanosphaera',
       'k__Archaea|p__Euryarchaeota|c__Methanobacteria|o__Methanobacteriales|f__Methanobacteriaceae|g__Methanosphaera|s__Methanosphaera_stadtmanae',
       'k__Archaea|p__Euryarchaeota|c__Thermoplasmata',
       ...
       'k__Viruses|p__Viruses_unclassified|c__Viruses_unclassified|o__Viruses_unclassified|f__Vi